In [ ]:
import pandas as pd
import json
import gdown
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Download the first dataset
file_id_1 = '1ge21L73uFSR-cC8bI0hXHhvM0EtV4PtG'
output_path_1 = 'movies_genres.json'
gdown.download(f'https://drive.google.com/uc?id={file_id_1}', output_path_1, quiet=False)

# Load the JSON data
with open(output_path_1, 'r') as f:
    movies_data = json.load(f)

# Convert to DataFrame with movies as rows and genres as columns
# Extract all unique genres
all_genres = sorted(list(set(genre for movie_genre_list in movies_data.values() for genre in movie_genre_list)))

# Create a list of dictionaries for the DataFrame
df_data = []
for movie_name, movie_info in movies_data.items():
    row = {'Movie': movie_name}
    for genre in all_genres:
        row[genre] = 1 if genre in movie_info else 0 # movie_info here is the list of genres, not a dict
    df_data.append(row)

movies_df = pd.DataFrame(df_data)
movies_df = movies_df.set_index('Movie')

print("DataFrame created successfully (first 5 rows):")
print(movies_df.head())

# Question 1: What is the unique number of genres present in the dataset?
unique_genres_count = len(all_genres)
print(f"\n1. Unique number of genres: {unique_genres_count}")

# Question 2: What is the cosine similarity between “Movie 1” and “Movie 10”?
# Ensure Movie 1 and Movie 10 exist in the DataFrame
if 'Movie 1' in movies_df.index and 'Movie 10' in movies_df.index:
    movie1_vector = movies_df.loc['Movie 1'].values.reshape(1, -1)
    movie10_vector = movies_df.loc['Movie 10'].values.reshape(1, -1)
    cosine_sim_movie1_movie10 = cosine_similarity(movie1_vector, movie10_vector)[0][0]
    print(f"\n2. Cosine similarity between Movie 1 and Movie 10: {cosine_sim_movie1_movie10:.4f}")
else:
    print("\nCould not find 'Movie 1' or 'Movie 10' in the dataset.")

# Question 3: Based on the cosine similarity scores, which of the following movies are the 5 most similar movies to “Movie 50”?
if 'Movie 50' in movies_df.index:
    movie50_vector = movies_df.loc['Movie 50'].values.reshape(1, -1)

    # Calculate cosine similarity with all other movies
    similarities = []
    for movie_name, movie_vector in movies_df.iterrows():
        if movie_name != 'Movie 50': # Exclude Movie 50 itself
            sim = cosine_similarity(movie50_vector, movie_vector.values.reshape(1, -1))[0][0]
            similarities.append((movie_name, sim))

    # Sort by similarity in descending order and get top 5
    similarities.sort(key=lambda x: x[1], reverse=True)
    top5_similar_movies = [movie[0] for movie in similarities[:5]]

    print(f"\n3. Top 5 most similar movies to Movie 50: {top5_similar_movies}")

    # Check which option matches
    options = {
        'a': ['Movie 84', 'Movie 9', 'Movie 6', 'Movie 72', 'Movie 59'],
        'b': ['Movie 84', 'Movie 9', 'Movie 6', 'Movie 43', 'Movie 52'],
        'c': ['Movie 72', 'Movie 1', 'Movie 2', 'Movie 61', 'Movie 76'],
        'd': ['Movie 36', 'Movie 58', 'Movie 14', 'Movie 66', 'Movie 27']
    }

    found_option = None
    for label, movies_list in options.items():
        # Sort both lists to ensure order doesn't matter for comparison
        if sorted(top5_similar_movies) == sorted(movies_list):
            found_option = label
            break

    if found_option:
        print(f"   The matching option is: {found_option.upper()}")
    else:
        print("   No matching option found among the choices.")

else:
    print("\nCould not find 'Movie 50' in the dataset.")

Downloading...
From: https://drive.google.com/uc?id=1ge21L73uFSR-cC8bI0hXHhvM0EtV4PtG
To: /content/movies_genres.json
100%|██████████| 8.26k/8.26k [00:00<00:00, 16.8MB/s]

DataFrame created successfully (first 5 rows):
         Action  Adventure  Animation  Comedy  Crime  Documentary  Drama  \
Movie                                                                      
Movie 1       0          0          1       0      0            0      1   
Movie 2       0          0          1       1      0            0      0   
Movie 3       0          0          0       0      1            0      0   
Movie 4       0          0          1       1      0            0      0   
Movie 5       0          0          0       0      1            0      1   

         Fantasy  Horror  Musical  Mystery  Romance  Science Fiction  \
Movie                                                                  
Movie 1        0       0        0        0        1                1   
Movie 2        0       1        0        0        0                0   
Movie 3        1       0        0        0        0                1   
Movie 4        0       0        0        0        0         

## Questions 4 and 5: User-User Similarity and Rating Prediction

In [ ]:
# Download the second dataset
file_id_2 = '1GpfbgVQ5JiBuYAiFt7T1NfUPaXysusPx'
output_path_2 = 'user_ratings.csv'
gdown.download(f'https://drive.google.com/uc?id={file_id_2}', output_path_2, quiet=False)

# Load the CSV data, using the first column as the index
ratings_df = pd.read_csv(output_path_2, index_col=0)

print("User Ratings DataFrame (Items as rows):")
display(ratings_df.head())

# Transpose the DataFrame to have Users as rows and Items as columns for User-User similarity
user_item_matrix = ratings_df.T
print("\nUser-Item Matrix (Users as rows, first 5 rows):")
display(user_item_matrix.head())

Downloading...
From: https://drive.google.com/uc?id=1GpfbgVQ5JiBuYAiFt7T1NfUPaXysusPx
To: /content/user_ratings.csv
100%|██████████| 271/271 [00:00<00:00, 737kB/s]

User Ratings DataFrame (Items as rows):


,User 2,User 3,User 4,User 5,User 6,User 7,User 8,User 9,User 10
User 1,,,,,,,,,
4,4,2,3,4,4,2,0,5,3
5,3,5,4,1,4,4,5,1,1
3,5,4,4,4,4,5,5,5,3
5,2,1,1,2,5,2,2,5,3
5,4,1,3,2,3,2,5,1,1



User-Item Matrix (Users as rows, first 5 rows):


User 1,4,5,3,5,5,2,3,3,3,0
User 2,4,3,5,2,4,0,4,5,1,4
User 3,2,5,4,1,1,3,3,2,4,4
User 4,3,4,4,1,3,5,3,5,1,2
User 5,4,1,4,2,2,1,2,5,2,4
User 6,4,4,4,5,3,1,4,2,4,2


Now, let's calculate the User-User similarity matrix using cosine similarity.

In [ ]:
# Calculate User-User Cosine Similarity Matrix
user_similarity_matrix = pd.DataFrame(cosine_similarity(user_item_matrix.fillna(0)),
                                      index=user_item_matrix.index,
                                      columns=user_item_matrix.index)

print("User-User Similarity Matrix (first 5 rows and columns):")
display(user_similarity_matrix.iloc[:5, :5])

User-User Similarity Matrix (first 5 rows and columns):


,User 2,User 3,User 4,User 5,User 6
User 2,1.000000,0.800342,0.857195,0.945093,0.860729
User 3,0.800342,1.000000,0.853647,0.792743,0.852335
User 4,0.857195,0.853647,1.000000,0.840676,0.781954
User 5,0.945093,0.792743,0.840676,1.000000,0.822330
User 6,0.860729,0.852335,0.781954,0.822330,1.000000


### Question 4: Which pair has the highest similarity score based on the User-User similarity matrix?

In [ ]:
# Question 4: Among the given pairs, which one has the highest similarity score?

pairs_to_check = {
    'a': ('User 8', 'User 10'),
    'b': ('User 1', 'User 9'),
    'c': ('User 2', 'User 5'),
    'd': ('User 7', 'User 3')
}

similarity_scores = {}
for label, (user1, user2) in pairs_to_check.items():
    # Ensure users exist and avoid self-similarity (which would be 1)
    if user1 in user_similarity_matrix.index and user2 in user_similarity_matrix.columns:
        score = user_similarity_matrix.loc[user1, user2]
        similarity_scores[label] = score
        print(f"Similarity between {user1} & {user2} (Option {label.upper()}): {score:.4f}")
    else:
        print(f"Could not find {user1} or {user2} in the similarity matrix for option {label.upper()}")

if similarity_scores:
    highest_score_label = max(similarity_scores, key=similarity_scores.get)
    print(f"\n4. The pair with the highest similarity score is Option {highest_score_label.upper()}: {pairs_to_check[highest_score_label][0]} & {pairs_to_check[highest_score_label][1]} with a score of {similarity_scores[highest_score_label]:.4f}")
else:
    print("No similarity scores could be calculated for the given options.")

Similarity between User 8 & User 10 (Option A): 0.6360
Could not find User 1 or User 9 in the similarity matrix for option B
Similarity between User 2 & User 5 (Option C): 0.9451
Similarity between User 7 & User 3 (Option D): 0.9721

4. The pair with the highest similarity score is Option D: User 7 & User 3 with a score of 0.9721


### Question 5: Predict the rating of item J for user-1 using User-User collaborative filtering.

In [ ]:
# Question 5: Predict the rating of item J for user-1 using User-User collaborative filtering

target_user = 'User 1'
target_item = 'J'

# Check if target_user exists in the similarity matrix index
if target_user not in user_similarity_matrix.index:
    print(f"Error: Target user '{target_user}' not found in the dataset.")
    print(f"Available users in the similarity matrix: {user_similarity_matrix.index.tolist()}")
else:
    # Get similarity scores for the target user with all other users
    similar_users = user_similarity_matrix.loc[target_user].drop(target_user)

    # Filter out users who haven't rated the target item or have 0 similarity
    # and sort by similarity in descending order
    users_who_rated_item = user_item_matrix[target_item].dropna().index

    relevant_similar_users = similar_users[similar_users.index.isin(users_who_rated_item)].sort_values(ascending=False)

    # Only consider users with positive similarity
    relevant_similar_users = relevant_similar_users[relevant_similar_users > 0]

    if relevant_similar_users.empty:
        print(f"Cannot predict rating for {target_item} by {target_user} as no similar users rated the item.")
    else:
        # Get the ratings of the target item from similar users
        similar_users_ratings = user_item_matrix.loc[relevant_similar_users.index, target_item]

        # Calculate the weighted sum of ratings
        weighted_sum = (similar_users_ratings * relevant_similar_users).sum()

        # Sum of absolute similarities for normalization
        sum_of_similarities = relevant_similar_users.sum()

        if sum_of_similarities == 0:
            predicted_rating = user_item_matrix.loc[target_user].mean() # Fallback to average rating if no valid similarities
            print(f"5. Predicted rating for {target_item} by {target_user} (fallback to user average): {predicted_rating:.4f}")
        else:
            predicted_rating = weighted_sum / sum_of_similarities
            print(f"\n5. Predicted rating for {target_item} by {target_user}: {predicted_rating:.4f}")

Error: Target user 'User 1' not found in the dataset.
Available users in the similarity matrix: ['User 2', 'User 3', 'User 4', 'User 5', 'User 6', 'User 7', 'User 8', 'User 9', 'User 10']
